In [1]:
from langgraph.graph import StateGraph , START, END
from typing import TypedDict, Literal, Annotated
from google import genai
from langchain_core.output_parsers import PydanticOutputParser
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

In [2]:
load_dotenv()

True

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation"
) 

model1 = ChatHuggingFace(llm = llm)
model2 = ChatHuggingFace(llm = llm)
model3 = ChatHuggingFace(llm = llm)


In [4]:
class Tweetstate(TypedDict):
    topic : str
    tweet : str
    evaluation : Literal["approved" , "needs_improvement"]
    feedback : str
    iteration : int
    max_iteration : int

In [5]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evalution : Literal["approved" , "needs_improvement"] = Field(..., description="Field evalution"),
    feedback : str = Field(..., description= "feedback for the tweet")

In [6]:
parser = PydanticOutputParser(pydantic_object=TweetEvaluation)

In [7]:
def  generate_tweet(state:Tweetstate):
    messages = [
        SystemMessage(content= "You are very funny and clear Twitter/X influncer."),
        HumanMessage(content=f"""
Write a short , original and hilarious tweet on the topic :"{state["topic"]}"
Rules :
- DO not use question answer format.
- MAX 280 Chararcter.
- use observation humour , irony, sarcase, or cultural refrences .
This is version {state['iteration'] + 1}.
""")
]
    response = model1.invoke(messages)

    if isinstance(response, list):
         response = response[0]

    return{
        "tweet" : response.content
    }

In [8]:
def evaluate_tweet(state:Tweetstate):
    messages = [
    SystemMessage(
        content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
    ),
    HumanMessage(
        content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?
2. Humor – Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness – Is it short, sharp, and scroll-stopping?
4. Virality Potential – Would people retweet or share it?
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- It ends with generic, throwaway, or deflating lines that weaken the humor (e.g., vague summaries)

Respond ONLY in this structured format:

evaluation: "approved" or "needs_improvement"
feedback: Brief paragraph explaining strengths and weaknesses
Respond ONLY in JSON format.
{parser.get_format_instructions()}
"""
    )
]
    result = model2.invoke(messages)
    structure_output = parser.parse(result.content)

    return{
        "evaluation" : structure_output.evalution,
        "feedback" : structure_output.feedback

    }



In [9]:
def optimize_tweet(state:Tweetstate):
    messages = [
    SystemMessage(
        content="You punch up tweets for virality and humor based on given feedback."
    ),
    HumanMessage(
        content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
"{state['tweet']}"

Rewrite it as a short, viral-worthy tweet.
Avoid Q&A style and stay under 280 characters
"""
    )
]

    response = model3.invoke(messages)
    iteration = state['iteration'] + 1

    return {
        "tweet" : response.content,
        "iteration" : iteration
    }


In [10]:
def route_evaluation(state: Tweetstate):
    if state["evaluation"] == "approved" or state["iteration"] >= state["max_iteration"]:
        return "approved"
    else:
        return "needs_improvement"


In [11]:
graph = StateGraph(Tweetstate)

graph.add_node("generate" , generate_tweet)
graph.add_node("evaluate" , evaluate_tweet)
graph.add_node("optimize" , optimize_tweet)

graph.add_edge(START, "generate")
graph.add_edge("generate", "evaluate")
graph.add_conditional_edges("evaluate" , route_evaluation,{"approved": END, "needs_improvement" : "optimize"})
graph.add_edge("optimize", "evaluate")

app = graph.compile()


In [12]:
initial_state = {
    "topic" : "Indian railway",
    "iteration" : 1,
    "max_iteration" : 5
}

app.invoke(initial_state)

c:\Users\Vivaan\OneDrive\Desktop\agentic_ai\myenv\lib\site-packages\pydantic\json_schema.py:2448: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='Field evalution'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


{'topic': 'Indian railway',
 'tweet': '"Indian Railways: Where \'delay expected\' is a tech upgrade. Because showing up late has never been so on-time. #IndianRailway #DelayedNotDisappointed"\n\n(Note: I\'ve condensed the tweet to focus on the punchline and made it more concise while maintaining the original humor and commentary.)',
 'evaluation': (FieldInfo(annotation=NoneType, required=True, description='Field evalution'),),
 'feedback': "The tweet showcases originality and clever wordplay, effectively poking fun at the Indian Railways' delayed schedules. The use of 'on-time' and 'tech upgrade' highlights the humor in a witty yet relatable way. The concise and well-formed tweet structure contributes to its punchiness, making it more likely to be shared and retweeted.",
 'iteration': 5,
 'max_iteration': 5}